# MandateGuard Evaluation

This notebook evaluates the detection pipeline across fraud archetypes and population scales.

In [ ]:
import sys
sys.path.insert(0, '.')

import httpx
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

SIMULATOR_URL = 'http://localhost:8081/simulate'
LEDGER_URL = 'http://localhost:8082/ledger'
DETECTION_URL = 'http://localhost:8084/detect'
FEATURE_URL = 'http://localhost:8083/features'

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Run Simulation

In [ ]:
resp = httpx.post(f'{SIMULATOR_URL}/run', timeout=120)
sim = resp.json()
print(f"Agents: {sim['totalAgents']}")
print(f"Transactions: {sim['totalTransactions']}")
print(f"Simulation time: {sim['simulationTimeMs']}ms")

## 2. Fetch & Analyze Transactions

In [ ]:
resp = httpx.get(f'{LEDGER_URL}/transactions/recent', params={'limit': 50000}, timeout=30)
txs = resp.json()

amounts = [float(tx['amount']) for tx in txs]
fraud_mask = [tx.get('isFraudLabel') or tx.get('fraudLabel', False) for tx in txs]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist([a for a, f in zip(amounts, fraud_mask) if not f], bins=50, alpha=0.7, label='Normal', color='#38bdf8')
axes[0].hist([a for a, f in zip(amounts, fraud_mask) if f], bins=50, alpha=0.7, label='Fraud', color='#ef4444')
axes[0].set_xlabel('Transaction Amount')
axes[0].set_ylabel('Count')
axes[0].set_title('Amount Distribution (Normal vs Fraud)')
axes[0].legend()
axes[0].set_yscale('log')

normal_count = sum(1 for f in fraud_mask if not f)
fraud_count = sum(1 for f in fraud_mask if f)
axes[1].bar(['Normal', 'Fraud'], [normal_count, fraud_count], color=['#38bdf8', '#ef4444'])
axes[1].set_ylabel('Count')
axes[1].set_title('Transaction Counts')

plt.tight_layout()
plt.show()

print(f"Total: {len(txs)}, Normal: {normal_count}, Fraud: {fraud_count}")
print(f"Fraud rate: {fraud_count/len(txs)*100:.2f}%")

## 3. Train Models & Run Detection

In [ ]:
resp = httpx.post(f'{DETECTION_URL}/train', timeout=60)
print(f"Training: {resp.json()}")

resp = httpx.get(f'{DETECTION_URL}/batch', params={'limit': 500}, timeout=120)
detections = resp.json()
print(f"Detected {len(detections)} agents")

## 4. Per-Archetype Precision/Recall/F1

In [ ]:
def classify_archetype(tx):
    if tx.get('isFraudLabel') or tx.get('fraudLabel'):
        amount = float(tx.get('amount', 0))
        if amount < 0.01:
            return 'micropayment_dos'
        if amount > 50.0:
            return 'velocity_anomaly'
    return 'normal'

agent_archetypes = defaultdict(set)
for tx in txs:
    from_id = tx.get('fromAgentId') or tx.get('from_agent_id')
    to_id = tx.get('toAgentId') or tx.get('to_agent_id')
    arch = classify_archetype(tx)
    if arch != 'normal':
        if from_id: agent_archetypes[from_id].add(arch)
        if to_id: agent_archetypes[to_id].add(arch)

archetypes = ['micropayment_dos', 'velocity_anomaly', 'sybil_cluster', 'collusion_ring', 'mandate_replay']
results = {}

for arch in archetypes:
    tp = fp = tn = fn = 0
    for d in detections:
        aid = d['agent_id']
        detected = d['is_anomaly']
        is_fraud = arch in agent_archetypes.get(aid, set())
        if detected and is_fraud: tp += 1
        elif detected and not is_fraud: fp += 1
        elif not detected and is_fraud: fn += 1
        else: tn += 1
    p = tp/(tp+fp) if (tp+fp) > 0 else 0
    r = tp/(tp+fn) if (tp+fn) > 0 else 0
    f = 2*p*r/(p+r) if (p+r) > 0 else 0
    results[arch] = {'precision': p, 'recall': r, 'f1': f, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(archetypes))
width = 0.25
ax.bar(x - width, [results[a]['precision'] for a in archetypes], width, label='Precision', color='#38bdf8')
ax.bar(x, [results[a]['recall'] for a in archetypes], width, label='Recall', color='#f97316')
ax.bar(x + width, [results[a]['f1'] for a in archetypes], width, label='F1', color='#22c55e')
ax.set_ylabel('Score')
ax.set_title('Per-Archetype Detection Performance')
ax.set_xticks(x)
ax.set_xticklabels([a.replace('_', '\n') for a in archetypes], fontsize=9)
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

for arch in archetypes:
    r = results[arch]
    print(f"{arch:<25} P={r['precision']:.3f}  R={r['recall']:.3f}  F1={r['f1']:.3f}")

## 5. Scalability Analysis

In [ ]:
pop_sizes = [500, 1000, 2000, 3000]
scalability_results = []

for pop in pop_sizes:
    print(f"Testing population {pop}...")
    try:
        resp = httpx.post(f'{SIMULATOR_URL}/run', json={'populationSize': pop}, timeout=120)
        sim_r = resp.json()
        resp = httpx.get(f'{DETECTION_URL}/batch', params={'limit': 200}, timeout=120)
        dets = resp.json()
        anomaly_rate = sum(1 for d in dets if d['is_anomaly']) / len(dets) if dets else 0
        scalability_results.append({
            'population': pop,
            'transactions': sim_r['totalTransactions'],
            'time_ms': sim_r['simulationTimeMs'],
            'anomaly_rate': anomaly_rate,
        })
    except Exception as e:
        print(f"  Error: {e}")

if scalability_results:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    pops = [r['population'] for r in scalability_results]

    axes[0].plot(pops, [r['transactions'] for r in scalability_results], 'o-', color='#38bdf8')
    axes[0].set_xlabel('Population')
    axes[0].set_ylabel('Transactions')
    axes[0].set_title('Transactions vs Population')

    axes[1].plot(pops, [r['time_ms'] for r in scalability_results], 's-', color='#f97316')
    axes[1].set_xlabel('Population')
    axes[1].set_ylabel('Time (ms)')
    axes[1].set_title('Simulation Time vs Population')

    axes[2].plot(pops, [r['anomaly_rate'] for r in scalability_results], '^-', color='#ef4444')
    axes[2].set_xlabel('Population')
    axes[2].set_ylabel('Anomaly Rate')
    axes[2].set_title('Detection Rate vs Population')

    plt.tight_layout()
    plt.show()

## 6. Graph Visualization

In [ ]:
try:
    import networkx as nx
    resp = httpx.get(f'{FEATURE_URL}/graph', params={'window_hours': 24}, timeout=30)
    graph_data = resp.json()

    G = nx.DiGraph()
    for node in graph_data.get('nodes', [])[:200]:
        G.add_node(node['id'][:8], degree=node['degree'])
    for edge in graph_data.get('edges', [])[:500]:
        s = edge['source'][:8] if edge['source'] in [n['id'] for n in graph_data.get('nodes', [])[:200]] else None
        t = edge['target'][:8] if edge['target'] in [n['id'] for n in graph_data.get('nodes', [])[:200]] else None
        if s and t:
            G.add_edge(s, t)

    if len(G.nodes()) > 0:
        fig, ax = plt.subplots(figsize=(12, 10))
        pos = nx.spring_layout(G, k=2/np.sqrt(len(G.nodes())), iterations=50, seed=42)
        degrees = [G.degree(n) for n in G.nodes()]
        max_d = max(degrees) if degrees else 1
        sizes = [50 + 200 * (d / max_d) for d in degrees]
        nx.draw_networkx_edges(G, pos, alpha=0.1, edge_color='#475569', ax=ax)
        nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=degrees, cmap=plt.cm.coolwarm, ax=ax)
        ax.set_title('Transaction Graph (spring layout)')
        plt.tight_layout()
        plt.show()
except ImportError:
    print("Install networkx: pip install networkx")